In [ ]:
import glfw
from OpenGL.GL import *
from OpenGL.GL.shaders import compileProgram, compileShader
import numpy as np
import glm
import sys

In [ ]:
# ==========================================
# 1. Shaders (Pipeline Moderno)
# ==========================================
VERTEX_SHADER = """
#version 330 core
layout (location = 0) in vec3 aPos;
layout (location = 1) in vec2 aTexCoord;

uniform mat4 model;
uniform mat4 view;
uniform mat4 projection;

out vec2 TexCoord;

void main()
{
    gl_Position = projection * view * model * vec4(aPos, 1.0);
    TexCoord = aTexCoord;
}
"""

FRAGMENT_SHADER = """
#version 330 core
out vec4 FragColor;
in vec2 TexCoord;
uniform sampler2D texture1;

void main()
{
    FragColor = texture(texture1, TexCoord); 
}
"""

In [ ]:
# ==========================================
# 2. Abstração de Objetos 3D e Câmera
# ==========================================
def load_obj(filename):
    vertices = []
    tex_coords = []
    final_vertices = []

    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('v '):
                parts = line.split()
                vertices.append((float(parts[1]), float(parts[2]), float(parts[3])))
            elif line.startswith('vt '):
                parts = line.split()
                tex_coords.append((float(parts[1]), float(parts[2])))
            elif line.startswith('f '):
                parts = line.split()[1:]
                # Assume que o modelo foi exportado com faces triangulares
                for vertex_data in parts:
                    indices = vertex_data.split('/')
                    v_idx = int(indices[0]) - 1 # .obj começa a contar do 1, Python do 0
                    
                    # Pega a posição (x, y, z)
                    x, y, z = vertices[v_idx]
                    
                    # Pega a textura (u, v) se existir
                    if len(indices) > 1 and indices[1] != '':
                        vt_idx = int(indices[1]) - 1
                        u, v = tex_coords[vt_idx]
                    else:
                        u, v = 0.0, 0.0
                        
                    final_vertices.extend([x, y, z, u, v])
                    
    return np.array(final_vertices, dtype=np.float32)

class GameObject:
    def __init__(self, obj_filepath=None):
        # Se passar um arquivo, carrega ele. Se não, usa o triângulo de teste.
        if obj_filepath:
            self.vertices = load_obj(obj_filepath)
        else:
            self.vertices = np.array([
                -0.5, -0.5, 0.0,       0.0, 0.0,
                 0.5, -0.5, 0.0,       1.0, 0.0,
                 0.0,  0.5, 0.0,       0.5, 1.0
            ], dtype=np.float32)
        
        # O número de vértices a desenhar agora é dinâmico (tamanho do array / 5 elementos por vértice)
        self.vertex_count = len(self.vertices) // 5
        
        self.vao = glGenVertexArrays(1)
        self.vbo = glGenBuffers(1)
        
        self.position = glm.vec3(0.0, 0.0, 0.0)
        self.rotation = glm.vec3(0.0, 0.0, 0.0)
        self.scale = glm.vec3(1.0, 1.0, 1.0)
        
        self.setup_mesh()

    def setup_mesh(self):
        glBindVertexArray(self.vao)
        glBindBuffer(GL_ARRAY_BUFFER, self.vbo)
        glBufferData(GL_ARRAY_BUFFER, self.vertices.nbytes, self.vertices, GL_STATIC_DRAW)
        
        glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, 5 * 4, ctypes.c_void_p(0))
        glEnableVertexAttribArray(0)
        
        glVertexAttribPointer(1, 2, GL_FLOAT, GL_FALSE, 5 * 4, ctypes.c_void_p(3 * 4))
        glEnableVertexAttribArray(1)
        
        glBindVertexArray(0)

    def get_model_matrix(self):
        model = glm.mat4(1.0)
        model = glm.translate(model, self.position)
        model = glm.rotate(model, glm.radians(self.rotation.x), glm.vec3(1.0, 0.0, 0.0))
        model = glm.rotate(model, glm.radians(self.rotation.y), glm.vec3(0.0, 1.0, 0.0))
        model = glm.rotate(model, glm.radians(self.rotation.z), glm.vec3(0.0, 0.0, 1.0))
        model = glm.scale(model, self.scale)
        return model

    def draw(self, shader_program):
        model_loc = glGetUniformLocation(shader_program, "model")
        glUniformMatrix4fv(model_loc, 1, GL_FALSE, glm.value_ptr(self.get_model_matrix()))
        
        glBindVertexArray(self.vao)
        # Substituímos o "3" fixo por self.vertex_count
        glDrawArrays(GL_TRIANGLES, 0, self.vertex_count) 
        glBindVertexArray(0)


class Camera:
    def __init__(self, position=glm.vec3(0.0, 1.5, 3.0)):
        self.position = position
        self.front = glm.vec3(0.0, 0.0, -1.0)
        self.up = glm.vec3(0.0, 1.0, 0.0)
        self.right = glm.vec3(1.0, 0.0, 0.0)
        self.world_up = glm.vec3(0.0, 1.0, 0.0)

        self.yaw = -90.0
        self.pitch = 0.0

        self.movement_speed = 2.5
        self.mouse_sensitivity = 0.1

        self.update_camera_vectors()

    def get_view_matrix(self):
        return glm.lookAt(self.position, self.position + self.front, self.up)

    def process_keyboard(self, direction, delta_time):
        velocity = self.movement_speed * delta_time
        if direction == "FORWARD":
            self.position += self.front * velocity
        if direction == "BACKWARD":
            self.position -= self.front * velocity
        if direction == "LEFT":
            self.position -= self.right * velocity
        if direction == "RIGHT":
            self.position += self.right * velocity

    def process_mouse_movement(self, xoffset, yoffset, constrain_pitch=True):
        xoffset *= self.mouse_sensitivity
        yoffset *= self.mouse_sensitivity

        self.yaw += xoffset
        self.pitch += yoffset

        if constrain_pitch:
            if self.pitch > 89.0:
                self.pitch = 89.0
            if self.pitch < -89.0:
                self.pitch = -89.0

        self.update_camera_vectors()

    def update_camera_vectors(self):
        front = glm.vec3()
        front.x = glm.cos(glm.radians(self.yaw)) * glm.cos(glm.radians(self.pitch))
        front.y = glm.sin(glm.radians(self.pitch))
        front.z = glm.sin(glm.radians(self.yaw)) * glm.cos(glm.radians(self.pitch))
        self.front = glm.normalize(front)
        
        self.right = glm.normalize(glm.cross(self.front, self.world_up))
        self.up = glm.normalize(glm.cross(self.right, self.front))

In [ ]:
# ==========================================
# 3. Variáveis Globais e Input
# ==========================================
is_wireframe = False
p_key_pressed = False # Para evitar que a tecla "fique segurada" mudando freneticamente
camera = Camera()
last_x = 400
last_y = 300
first_mouse = True
delta_time = 0.0
last_frame = 0.0

def mouse_callback(window, xpos, ypos):
    global last_x, last_y, first_mouse, camera

    if first_mouse:
        last_x = xpos
        last_y = ypos
        first_mouse = False

    xoffset = xpos - last_x
    yoffset = last_y - ypos 

    last_x = xpos
    last_y = ypos

    camera.process_mouse_movement(xoffset, yoffset)

def process_input(window):
    global camera, delta_time, is_wireframe, p_key_pressed
    if glfw.get_key(window, glfw.KEY_ESCAPE) == glfw.PRESS:
        glfw.set_window_should_close(window, True)

    if glfw.get_key(window, glfw.KEY_W) == glfw.PRESS:
        camera.process_keyboard("FORWARD", delta_time)
    if glfw.get_key(window, glfw.KEY_S) == glfw.PRESS:
        camera.process_keyboard("BACKWARD", delta_time)
    if glfw.get_key(window, glfw.KEY_A) == glfw.PRESS:
        camera.process_keyboard("LEFT", delta_time)
    if glfw.get_key(window, glfw.KEY_D) == glfw.PRESS:
        camera.process_keyboard("RIGHT", delta_time)
    if glfw.get_key(window, glfw.KEY_P) == glfw.PRESS:
        if not p_key_pressed: # Só muda no instante que você aperta
            is_wireframe = not is_wireframe
            if is_wireframe:
                glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
            else:
                glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)
            p_key_pressed = True
    elif glfw.get_key(window, glfw.KEY_P) == glfw.RELEASE:
        p_key_pressed = False


In [ ]:
# ==========================================
# 4. Inicialização e Loop Principal
# ==========================================
def main():
    global delta_time, last_frame, camera
    
    if not glfw.init():
        sys.exit()
        
    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)

    window = glfw.create_window(800, 600, "Pescaria Mágica 3D", None, None)
    if not window:
        glfw.terminate()
        sys.exit()

    glfw.make_context_current(window)
    
    glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)
    glfw.set_cursor_pos_callback(window, mouse_callback)
    
    shader = compileProgram(
        compileShader(VERTEX_SHADER, GL_VERTEX_SHADER),
        compileShader(FRAGMENT_SHADER, GL_FRAGMENT_SHADER)
    )
    
    glEnable(GL_DEPTH_TEST)
    glClearColor(0.05, 0.05, 0.15, 1.0) 

    dummy_object = GameObject()
    dummy_object.position = glm.vec3(0.0, 0.0, 0.0)

    while not glfw.window_should_close(window):
        current_frame = glfw.get_time()
        delta_time = current_frame - last_frame
        last_frame = current_frame

        process_input(window)

        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        glUseProgram(shader)

        projection = glm.perspective(glm.radians(45.0), 800.0 / 600.0, 0.1, 100.0)
        view = camera.get_view_matrix()

        proj_loc = glGetUniformLocation(shader, "projection")
        view_loc = glGetUniformLocation(shader, "view")
        glUniformMatrix4fv(proj_loc, 1, GL_FALSE, glm.value_ptr(projection))
        glUniformMatrix4fv(view_loc, 1, GL_FALSE, glm.value_ptr(view))

        dummy_object.draw(shader)

        glfw.swap_buffers(window)
        glfw.poll_events()

    glfw.terminate()

# Executa o código
if __name__ == '__main__':
    main()